In [1]:
# R4_discrimination.ipynb  (revision stage 3 - discrimination, addresses M7)
# Same-format discrimination table for ALL THREE datasets (not just GMSC):
# mean +/- sd AUC, DeLong per-split (stump vs logit), Wilcoxon across splits,
# and an unconstrained XGBoost ceiling. Parity is claimed via a TOST equivalence
# test with a pre-set margin on AUC (not merely 'failed to reject'). Binning + IV
# INSIDE each training fold (M6). Exact repeat counts are recorded per dataset.
# Single-cell; greyscale seaborn; dpi600 png+pdf; no captions.

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
import xgboost as xgb
warnings.filterwarnings('ignore')

SEED=42; IV_THRESH=0.02; EQ_MARGIN=0.01   # AUC equivalence margin (+/-0.01)
sns.set_theme(style='whitegrid', context='paper'); sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor':'0.2','grid.color':'0.85','font.size':10})
ROOT=os.path.abspath(os.path.join(os.getcwd(),'..')); DATA=os.path.join(ROOT,'data')
TABLES=os.path.join(ROOT,'results','tables'); FIGS=os.path.join(ROOT,'results','figures')
def savefig(fig,name):
    for e in ('png','pdf'): fig.savefig(os.path.join(FIGS,name+'.'+e),dpi=600,bbox_inches='tight')
    plt.close(fig)
SPECIAL=[-7,-8,-9]

def ks(yt,s): fpr,tpr,_=roc_curve(yt,s); return float(np.max(tpr-fpr))
def delong(yt,s1,s2):
    yt=np.asarray(yt); pos=yt==1; neg=~pos; m=pos.sum(); nn=neg.sum()
    def st(s):
        sp,sn=s[pos],s[neg]; V10=np.array([(np.sum(sn<x)+.5*np.sum(sn==x))/nn for x in sp])
        V01=np.array([(np.sum(sp>x)+.5*np.sum(sp==x))/m for x in sn]); return V10.mean(),V10,V01
    a1,x1,y1=st(s1); a2,x2,y2=st(s2); S=np.cov(np.vstack([x1,x2]))/m+np.cov(np.vstack([y1,y2]))/nn
    v=S[0,0]+S[1,1]-2*S[0,1]
    if v<=0: return a1,a2,np.nan,np.nan
    z=(a1-a2)/np.sqrt(v); return a1,a2,float(z),float(2*(1-stats.norm.cdf(abs(z))))
def tost_auc(diffs,margin=EQ_MARGIN):
    d=np.asarray(diffs); m=d.mean(); se=d.std(ddof=1)/np.sqrt(len(d))
    return max(stats.t.sf((m+margin)/se,len(d)-1), stats.t.sf((margin-m)/se,len(d)-1))

# ---- generic in-fold binning (numeric monotone + categorical + special codes) ----
def make_binner(cat_cols,num_cols):
    def tree_edges(x,yv):
        m=~x.isin(SPECIAL)&x.notna(); xx=x[m].values.reshape(-1,1).astype(float); yy=yv[m.values]
        if len(np.unique(xx))<=2: return []
        t=DecisionTreeClassifier(max_leaf_nodes=6,min_samples_leaf=int(0.05*len(xx)),random_state=SEED)
        t.fit(xx,yy); return sorted(float(v) for v in t.tree_.threshold[t.tree_.feature>=0])
    def build(Xtr,ytr):
        TG=int((ytr==0).sum()); TB=int((ytr==1).sum()); min_n=max(int(0.02*len(ytr)),10); maps={}; iv={}
        def woe(g,b): return float(np.log(((b+0.5)/(TB+0.5))/((g+0.5)/(TG+0.5))))
        for c in cat_cols:
            recs=[]; 
            for lv in sorted(Xtr[c].dropna().unique()):
                sel=(Xtr[c]==lv).values; recs.append({'kind':'lvl','lv':lv,'good':int((ytr[sel]==0).sum()),'bad':int((ytr[sel]==1).sum())})
            keep=[]; og=ob=0
            for r in recs:
                if r['good']+r['bad']<min_n: og+=r['good']; ob+=r['bad']
                else: keep.append(r)
            if og+ob>0: keep.append({'kind':'other','good':og,'bad':ob})
            for r in keep: r['woe']=woe(r['good'],r['bad'])
            maps[c]=keep; iv[c]=float(np.sum([((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in keep]))
        for c in num_cols:
            e=tree_edges(Xtr[c],ytr)
            def tbl(e):
                cuts=[-np.inf]+list(e)+[np.inf]; m=~Xtr[c].isin(SPECIAL)&Xtr[c].notna()
                idx=pd.cut(Xtr[c][m],cuts,labels=False,include_lowest=True); rr=[]
                for b in range(len(cuts)-1):
                    sel=idx==b; g=int((ytr[m.values][sel]==0).sum()); bd=int((ytr[m.values][sel]==1).sum())
                    if g+bd>0: rr.append({'kind':'num','lo':cuts[b],'hi':cuts[b+1],'good':g,'bad':bd})
                return rr
            while e:
                rr=tbl(e); br=np.array([r['bad']/(r['good']+r['bad']) for r in rr])
                if len(br)<=2 or np.all(np.diff(br)>=-1e-12) or np.all(np.diff(br)<=1e-12): break
                d=np.diff(br); s=np.sign(np.nansum(d)); e.pop(int(np.argmax(-s*d)))
            recs=tbl(e)
            for code in SPECIAL:
                sel=(Xtr[c]==code).values; g=int((ytr[sel]==0).sum()); b=int((ytr[sel]==1).sum())
                if g+b>0: recs.append({'kind':'code','code':code,'good':g,'bad':b})
            seln=Xtr[c].isna().values; gn=int((ytr[seln]==0).sum()); bn=int((ytr[seln]==1).sum())
            if gn+bn>0: recs.append({'kind':'nan','good':gn,'bad':bn})
            keep=[]; og=ob=0
            for r in recs:
                if r['kind'] in('code','nan') and r['good']+r['bad']<min_n: og+=r['good']; ob+=r['bad']
                else: keep.append(r)
            if og+ob>0: keep.append({'kind':'other','good':og,'bad':ob})
            for r in keep: r['woe']=woe(r['good'],r['bad'])
            maps[c]=keep; iv[c]=float(np.sum([((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in keep]))
        keep=[c for c in (cat_cols+num_cols) if iv[c]>=IV_THRESH]
        return maps,keep
    def to_idx(x,recs):
        idx=np.full(len(x),-1,int)
        for p,r in enumerate(recs):
            if r['kind']=='num': sel=(~x.isin(SPECIAL)).values&x.notna().values&(x>r['lo']).values&(x<=r['hi']).values; idx[sel]=p
            elif r['kind']=='lvl': idx[(x==r['lv']).values]=p
            elif r['kind']=='code': idx[(x==r['code']).values]=p
            elif r['kind']=='nan': idx[x.isna().values]=p
        oth=[p for p,r in enumerate(recs) if r['kind']=='other']; idx[idx==-1]=oth[0] if oth else 0; return idx
    def woe_mat(Xdf,maps,keep):
        return np.column_stack([np.array([r['woe'] for r in maps[c]])[to_idx(Xdf[c],maps[c])] for c in keep])
    return build,woe_mat

def run(name,df,TARGET,cat_cols,num_cols,n_split,xraw_cols):
    build,woe_mat=make_binner(cat_cols,num_cols); y=df[TARGET].values.astype(int)
    if len(xraw_cols)==0:
        Xraw=None
    else:
        Xraw=df[xraw_cols].replace(SPECIAL,np.nan); Xraw=Xraw.fillna(Xraw.median(numeric_only=True))
    rows=[]; dl=[]; t0=time.time()
    for sp in range(n_split):
        idx_tr,idx_te=train_test_split(np.arange(len(df)),test_size=0.3,stratify=y,random_state=SEED+sp)
        Xtr=df.iloc[idx_tr]; ytr=y[idx_tr]; Xte=df.iloc[idx_te]; yte=y[idx_te]
        maps,keep=build(Xtr,ytr)
        if len(keep)<2: continue
        Wtr=woe_mat(Xtr,maps,keep); Wte=woe_mat(Xte,maps,keep)
        lr=LogisticRegression(max_iter=1000).fit(Wtr,ytr); s_lr=lr.decision_function(Wte)
        gb=GradientBoostingClassifier(max_depth=1,learning_rate=0.3,n_estimators=300,random_state=SEED+sp).fit(Wtr,ytr)
        s_gb=gb.decision_function(Wte)
        auc_x=np.nan
        if Xraw is not None:
            xg=xgb.XGBClassifier(max_depth=4,n_estimators=300,learning_rate=0.1,subsample=0.9,
                                 colsample_bytree=0.9,n_jobs=4,eval_metric='logloss',random_state=SEED+sp)
            xg.fit(Xraw.iloc[idx_tr].values,ytr); auc_x=roc_auc_score(yte,xg.predict_proba(Xraw.iloc[idx_te].values)[:,1])
        a_gb,a_lr,z,pv=delong(yte,s_gb,s_lr)
        rows.append([sp,roc_auc_score(yte,s_lr),roc_auc_score(yte,s_gb),auc_x,ks(yte,s_lr),ks(yte,s_gb)])
        dl.append([sp,a_gb-a_lr,z,pv])
    R=pd.DataFrame(rows,columns=['sp','auc_lr','auc_gb','auc_xgb','ks_lr','ks_gb'])
    D=pd.DataFrame(dl,columns=['sp','auc_diff','z','p'])
    w=stats.wilcoxon(R['auc_gb'],R['auc_lr'])
    tost=tost_auc(R['auc_gb']-R['auc_lr'])
    print('\n===== %s  (%d splits, %.0fs) ====='%(name,len(R),time.time()-t0))
    print('logit AUC=%.4f\u00b1%.4f | stump AUC=%.4f\u00b1%.4f | xgb=%.4f'%(
        R['auc_lr'].mean(),R['auc_lr'].std(),R['auc_gb'].mean(),R['auc_gb'].std(),R['auc_xgb'].mean()))
    print('mean diff(stump-logit)=%+.5f | Wilcoxon p=%.3g | DeLong frac p<.05=%.2f | TOST(|d|<%.3f) p=%.3g %s'%(
        (R['auc_gb']-R['auc_lr']).mean(),w.pvalue,(D['p']<0.05).mean(),EQ_MARGIN,tost,'(equiv)' if tost<0.05 else '(NOT equiv)'))
    R.round(6).to_csv(os.path.join(TABLES,'R4_%s_auc.csv'%name),index=False)
    return dict(name=name,n=len(R),auc_lr=R['auc_lr'].mean(),auc_lr_sd=R['auc_lr'].std(),
                auc_gb=R['auc_gb'].mean(),auc_gb_sd=R['auc_gb'].std(),auc_xgb=R['auc_xgb'].mean(),
                diff=(R['auc_gb']-R['auc_lr']).mean(),wilcox=w.pvalue,delong_frac=(D['p']<0.05).mean(),tost=tost), R

# ---- GMSC ----
g=pd.read_parquet(os.path.join(DATA,'gmsc_clean.parquet')); gT='SeriousDlqin2yrs'
gnum=[c for c in g.columns if c!=gT]
res_g,Rg=run('GMSC',g,gT,[],gnum,50,gnum)
# ---- German ----
cols=['Status_checking','Duration','Credit_history','Purpose','Credit_amount','Savings','Employment_since',
      'Installment_rate','Personal_status_sex','Other_debtors','Residence_since','Property','Age',
      'Other_installment','Housing','Existing_credits','Job','Num_liable','Telephone','Foreign_worker','target']
gm=pd.read_csv(os.path.join(DATA,'german.data'),sep=' ',header=None,names=cols); gm['target']=(gm['target']==2).astype(int)
gmcat=['Status_checking','Credit_history','Purpose','Savings','Employment_since','Personal_status_sex',
       'Other_debtors','Property','Other_installment','Housing','Job','Telephone','Foreign_worker']
gmnum=['Duration','Credit_amount','Installment_rate','Residence_since','Age','Existing_credits','Num_liable']
res_gm,Rgm=run('German',gm,'target',gmcat,gmnum,50,[])  # xgb skipped (mixed categorical types)
# ---- HELOC ----
h=pd.read_csv(os.path.join(DATA,'heloc.csv')); h['RiskPerformance']=(h['RiskPerformance']=='Bad').astype(int)
hf=[c for c in h.columns if c!='RiskPerformance']; allmiss=(h[hf]==-9).all(axis=1); h=h[~allmiss].reset_index(drop=True)
res_h,Rh=run('HELOC',h,'RiskPerformance',[],hf,50,hf)

# ---- combined table + figure ----
summary=pd.DataFrame([res_g,res_gm,res_h])
summary.to_csv(os.path.join(TABLES,'R4_discrimination_summary.csv'),index=False)
print('\n=== Discrimination summary (all datasets) ===')
print(summary.round(4).to_string(index=False))

fig,ax=plt.subplots(figsize=(7.6,4.6))
names=['GMSC','German','HELOC']; x=np.arange(len(names))
for Rr,offset,c,lab,key in [(Rg,-0.2,'0.4','logistic-WoE','auc_lr'),(Rg,0.0,'0.65','stump-boost','auc_gb')]:
    pass
datasets=[Rg,Rgm,Rh]
lr_m=[d['auc_lr'].mean() for d in datasets]; lr_s=[d['auc_lr'].std() for d in datasets]
gb_m=[d['auc_gb'].mean() for d in datasets]; gb_s=[d['auc_gb'].std() for d in datasets]
ax.bar(x-0.17,lr_m,0.34,yerr=lr_s,color='0.4',edgecolor='0.0',capsize=3,label='logistic-WoE')
ax.bar(x+0.17,gb_m,0.34,yerr=gb_s,color='0.7',edgecolor='0.0',capsize=3,label='stump-boost')
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylabel('AUC'); ax.set_ylim(0.7,0.88)
ax.legend(frameon=True,facecolor='white',edgecolor='0.5'); savefig(fig,'R4_discrimination')
print('\nSaved: R4_discrimination, per-dataset AUC CSVs, R4_discrimination_summary')
print('\n================ R4 HEADLINE ================')
for r in [res_g,res_gm,res_h]:
    print('%-7s: logit %.3f vs stump %.3f (diff %+.4f) Wilcoxon p=%.2g TOST p=%.2g %s'%(
        r['name'],r['auc_lr'],r['auc_gb'],r['diff'],r['wilcox'],r['tost'],'equiv' if r['tost']<0.05 else 'NOT equiv'))
print('\nR4 complete.')



===== GMSC  (50 splits, 467s) =====
logit AUC=0.8459±0.0031 | stump AUC=0.8467±0.0032 | xgb=0.8638
mean diff(stump-logit)=+0.00080 | Wilcoxon p=1.78e-15 | DeLong frac p<.05=0.52 | TOST(|d|<0.010) p=5.3e-70 (equiv)



===== German  (50 splits, 13s) =====
logit AUC=0.7773±0.0213 | stump AUC=0.7704±0.0223 | xgb=nan
mean diff(stump-logit)=-0.00688 | Wilcoxon p=5.58e-06 | DeLong frac p<.05=0.10 | TOST(|d|<0.010) p=0.0073 (equiv)



===== HELOC  (50 splits, 75s) =====
logit AUC=0.8048±0.0071 | stump AUC=0.8032±0.0069 | xgb=0.7904
mean diff(stump-logit)=-0.00163 | Wilcoxon p=3.09e-12 | DeLong frac p<.05=0.26 | TOST(|d|<0.010) p=6.23e-44 (equiv)

=== Discrimination summary (all datasets) ===
  name  n  auc_lr  auc_lr_sd  auc_gb  auc_gb_sd  auc_xgb    diff  wilcox  delong_frac   tost
  GMSC 50  0.8459     0.0031  0.8467     0.0032   0.8638  0.0008     0.0         0.52 0.0000
German 50  0.7773     0.0213  0.7704     0.0223      NaN -0.0069     0.0         0.10 0.0073
 HELOC 50  0.8048     0.0071  0.8032     0.0069   0.7904 -0.0016     0.0         0.26 0.0000



Saved: R4_discrimination, per-dataset AUC CSVs, R4_discrimination_summary

================ R4 HEADLINE ================
GMSC   : logit 0.846 vs stump 0.847 (diff +0.0008) Wilcoxon p=1.8e-15 TOST p=5.3e-70 equiv
German : logit 0.777 vs stump 0.770 (diff -0.0069) Wilcoxon p=5.6e-06 TOST p=0.0073 equiv
HELOC  : logit 0.805 vs stump 0.803 (diff -0.0016) Wilcoxon p=3.1e-12 TOST p=6.2e-44 equiv

R4 complete.
